### Power Network and Statistical Package

In [11]:
import os, shutil, random
import numpy as np
import networkx as nx
import pandas as pd
import torch
import copy
import cvxpy as cp

#Pandapower Package
import pandapower as pp
import pandapower.networks as pn
import pandapower.plotting as plot
import pandapower.diagnostic as diagnostic

from pandapower.powerflow import LoadflowNotConverged
from pandapower.diagnostic import diagnostic
from pandapower.control import ConstControl
from pandapower.timeseries import DFData, OutputWriter, run_timeseries
from pandapower.pypower.makeYbus import makeYbus

import matplotlib.pyplot as plt
from scipy.stats import norm
import scipy.linalg
import scipy.sparse as sp
from scipy.stats import gaussian_kde
from itertools import product
from tqdm import tqdm

In [13]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

### Define the Function to check N-1 contingency criterion with the test case

In [16]:
# Define the function to check whether the network meet the N-1 contingency criterion
def check_n_1_contingency(net):
    critical_elements = []
    
    # Backup original network
    original_net = net.deepcopy()
    
    # Test line outages
    for line in net.line.index:
        net_copy = original_net.deepcopy()
        net_copy.line.at[line, "in_service"] = False  # Remove one line
        try:
            pp.runpp(net_copy, algorithm="nr")
        except pp.powerflow.LoadflowNotConverged:
            critical_elements.append(f"Line {line} outage causes failure.")
            continue
        
        # Check for overloads
        if any(net_copy.res_line.loading_percent > 100):
            critical_elements.append(f"Line {line} outage causes overloads.")

    # Test generator outages
    for gen in net.gen.index:
        net_copy = original_net.deepcopy()
        net_copy.gen.at[gen, "in_service"] = False  # Remove one generator
        try:
            pp.runpp(net_copy, algorithm="nr")
        except pp.powerflow.LoadflowNotConverged:
            critical_elements.append(f"Generator {gen} outage causes failure.")
            continue

        # Check for voltage violations
        if any((net_copy.res_bus.vm_pu < 0.95) | (net_copy.res_bus.vm_pu > 1.05)):
            critical_elements.append(f"Generator {gen} outage causes voltage issues.")

    return critical_elements

### Load the network test case

In [19]:
# Load the IEEE 24-bus reliability test system 
net = pn.case24_ieee_rts()
pp.runpp(net)

In [20]:
# Tag buses for reference in the information layer
for i in net.bus.index:
    net.bus.at[i, "tag"] = f"Bus-{i}"

### The Function to reinforce the network to meet the N-1 criterion

In [ ]:
# Function to reinforce the network based on contingency violations
def reinforce_network(net):
    original_net = net.deepcopy()
    
    # Identify problematic lines
    overloaded_lines = []
    for line in net.line.index:
        net_copy = original_net.deepcopy()
        net_copy.line.at[line, "in_service"] = False  # Simulate line outage
        try:
            pp.runpp(net_copy)
        except pp.powerflow.LoadflowNotConverged:
            overloaded_lines.append(line)
            continue
        if any(net_copy.res_line.loading_percent > 100):
            overloaded_lines.append(line)
    
    # Add parallel lines to overloaded lines
    for line in overloaded_lines:
        from_bus = net.line.loc[line, "from_bus"]
        to_bus = net.line.loc[line, "to_bus"]
        print(f"Adding parallel line between Bus {from_bus} and Bus {to_bus} to mitigate overload.")
        pp.create_line_from_parameters(net, from_bus=from_bus, to_bus=to_bus, 
                                       length_km=1.0, r_ohm_per_km=0.05, 
                                       x_ohm_per_km=0.1, c_nf_per_km=0, 
                                       max_i_ka=1.5)  # Higher capacity
    
    # Identify problematic generators
    problematic_gens = []
    for gen in net.gen.index:
        net_copy = original_net.deepcopy()
        net_copy.gen.at[gen, "in_service"] = False  # Simulate generator outage
        try:
            pp.runpp(net_copy)
        except pp.powerflow.LoadflowNotConverged:
            problematic_gens.append(gen)
            continue
        if any((net_copy.res_bus.vm_pu < 0.95) | (net_copy.res_bus.vm_pu > 1.05)):
            problematic_gens.append(gen)

    # Increase generator capacities to provide redundancy
    for gen in problematic_gens:
        net.gen.at[gen, "p_mw"] *= 1.2  # Increase power generation by 20%
        print(f"Increasing capacity of Generator {gen} to improve voltage stability.")

    return net

# Reinforce the network
net = reinforce_network(net)

# Run contingency check again
pp.runpp(net)
print("Reinforced system power flow successful.")

# Check if violations still exist
violations = check_n_1_contingency(net)
if violations:
    print("System still has issues under N-1 contingency.")
    for v in violations:
        print(v)
else:
    print("The RTS 24-bus system satisfies the N-1 contingency criterion!")

In [ ]:
import copy
import numpy as np
import pandapower as pp

def check_n_1_contingency_pf(net, vmin=0.95, vmax=1.05, line_loading_limit=100.0):
    """
    N-1 check using power flow (fixed dispatch).
    Returns a list of human-readable violation strings and a structured dict.
    """
    critical = []
    details = {"line_outage": [], "gen_outage": []}

    original_net = copy.deepcopy(net)  

    # ---- Line outages ----
    for line in original_net.line.index:
        net_copy = copy.deepcopy(original_net)
        net_copy.line.at[line, "in_service"] = False
        try:
            pp.runpp(net_copy, algorithm="nr")
        except pp.powerflow.LoadflowNotConverged:
            critical.append(f"Line {line} outage causes failure (PF not converged).")
            details["line_outage"].append({"outaged_line": int(line), "reason": "pf_not_converged"})
            continue

        # survivor line overloads
        over_mask = net_copy.res_line.loading_percent > line_loading_limit
        overloaded = list(net_copy.res_line.index[over_mask])
        if line in overloaded:
            overloaded.remove(line)

        vm = net_copy.res_bus.vm_pu.values
        v_ok = (vm.min() >= vmin) and (vm.max() <= vmax)

        if overloaded or not v_ok:
            msg = []
            if overloaded:
                msg.append(f"overloads on {overloaded}")
            if not v_ok:
                msg.append(f"voltage min={vm.min():.3f}, max={vm.max():.3f}")
            critical.append(f"Line {line} outage -> " + "; ".join(msg))
            details["line_outage"].append({
                "outaged_line": int(line),
                "overloaded_lines": overloaded,
                "min_vm": float(vm.min()),
                "max_vm": float(vm.max()),
                "reason": "limits"
            })

    # ---- Gen outages ----
    for g in original_net.gen.index:
        net_copy = copy.deepcopy(original_net)
        net_copy.gen.at[g, "in_service"] = False
        try:
            pp.runpp(net_copy, algorithm="nr")
        except pp.powerflow.LoadflowNotConverged:
            critical.append(f"Generator {g} outage causes failure (PF not converged).")
            details["gen_outage"].append({"outaged_gen": int(g), "reason": "pf_not_converged"})
            continue

        vm = net_copy.res_bus.vm_pu.values
        v_ok = (vm.min() >= vmin) and (vm.max() <= vmax)
        if not v_ok:
            critical.append(f"Generator {g} outage causes voltage issues (min={vm.min():.3f}, max={vm.max():.3f}).")
            details["gen_outage"].append({
                "outaged_gen": int(g),
                "min_vm": float(vm.min()),
                "max_vm": float(vm.max()),
                "reason": "voltage"
            })

        over_mask = net_copy.res_line.loading_percent > line_loading_limit
        overloaded = list(net_copy.res_line.index[over_mask])
        if overloaded:
            critical.append(f"Generator {g} outage causes line overloads: {overloaded}.")
            # merge with existing record if any
            found = next((d for d in details["gen_outage"] if d.get("outaged_gen")==int(g)), None)
            if found:
                found["overloaded_lines"] = overloaded
                found["reason"] = "limits"
            else:
                details["gen_outage"].append({
                    "outaged_gen": int(g),
                    "overloaded_lines": overloaded,
                    "reason": "limits"
                })

    details["ok"] = (len(critical) == 0)
    return critical, details

### Built PMU with the Loaded Network Grid

In [ ]:
def get_single_bus_pmu_measurement(net, bus_index):
    """
    Extract PMU measurement for a single bus:
    - Voltage magnitude (p.u.)
    - Voltage angle (degrees)
    - Active power injection (MW)
    - Reactive power injection (MVAR)
    
    Returns a dictionary with all relevant fields.
    """
    vm = float(net.res_bus.vm_pu.at[bus_index])
    va = float(net.res_bus.va_degree.at[bus_index])
    p = float(net.res_bus.p_mw.at[bus_index])
    q = float(net.res_bus.q_mvar.at[bus_index])
    
    return {
        'bus': bus_index,
        'vm_pu': vm,
        'va_degree': va,
        'p_mw': p,
        'q_mvar': q
    }


def pdc(net, pmu_buses):
    """
    Simulate the Phasor Data Concentrator (PDC):
    Collects PMU data from all buses listed in `pmu_buses`.
    Returns a list of measurement dictionaries.
    """
    pmu_data = []
    for bus in pmu_buses:
        measurement = get_single_bus_pmu_measurement(net, bus)
        pmu_data.append(measurement)
    return pmu_data


# Example usage
pmu_buses = net.bus.index.tolist()  # simulate PMU on all buses
pmu_data_frame = pdc(net, pmu_buses)

# Optional: convert to a table (like your screenshot)
import pandas as pd
pmu_df = pd.DataFrame(pmu_data_frame)
print(pmu_df)

In [ ]:
 net.res_bus

### Information Layer Construct

In [ ]:
# Define sensors at selected buses
pmu_buses = [1, 3, 7, 15]
sensor_data = {}

for bus in pmu_buses:
    sensor_data[f"Bus-{bus}"] = {
        "type": "PMU",
        "vm_pu_true": None,
        "vm_pu_measured": None,
        "measurement_delay": 1,  # in timesteps
        "communication_error": 0.01  # 1% measurement noise
    }

In [ ]:
def simulate_info_layer(net, sensor_data):
    for bus_tag, data in sensor_data.items():
        bus_idx = int(bus_tag.split("-")[1])
        true_value = net.res_bus.vm_pu.at[bus_idx]
        noise = np.random.normal(0, data["communication_error"])
        sensor_data[bus_tag]["vm_pu_true"] = true_value
        sensor_data[bus_tag]["vm_pu_measured"] = true_value + noise

### Apply the cyber attack by injecting false data

In [ ]:
def inject_false_data(pmu_data, attack_buses, voltage_offset=0.05, angle_offset=2.0):
    """
    Inject false data at specified bus indices.
    voltage_offset: additive error in per unit (p.u.)
    angle_offset: additive error in degrees
    """
    pmu_data_attacked = pmu_data.copy()
    # Create copies to avoid modifying original
    voltage_attacked = np.copy(pmu_data_attacked['voltage'])
    angle_attacked = np.copy(pmu_data_attacked['angle'])
    
    for idx in attack_buses:
        voltage_attacked[idx] += voltage_offset  # add offset
        angle_attacked[idx] += angle_offset
    pmu_data_attacked['voltage'] = voltage_attacked
    pmu_data_attacked['angle'] = angle_attacked
    return pmu_data_attacked

# Assume attackers compromise bus 5, 12, and 23 
attack_buses = [5, 12, 23]
pmu_data_attacked = inject_false_data(pmu_data_true, attack_buses)

In [ ]:
def compute_voltage_deviation(net, baseline_vm_pu):
    deviations = abs(net.res_bus.vm_pu - baseline_vm_pu)
    return deviations.mean()

In [ ]:
def compute_custom_resilience_metric(system_loss, recovery_time, max_loss=1.0):
    # Normalize values between 0 and 1
    L_norm = system_loss / max_loss
    T_norm = recovery_time / 10  # assume 10 is worst case

    # Resilience = 1 - (weighted penalty)
    return 1 - (0.6 * L_norm + 0.4 * T_norm)

### The Resilience Metric
Evaluate resilience via:
1.Time to recovery

Extent of degraded performance

Cascading failure depth

Resilience Index (NEDI, R_cyber, etc.)

###  Run Time-Series Simulations with Events
Define multiple time steps to simulate pre-attack, attack, recovery periods

In [ ]:
T = 10
baseline_vm_pu = None

for t in range(T):
    if t == 0:
        pp.runpp(net)
        baseline_vm_pu = net.res_bus.vm_pu.copy()

    if t == 3:
        apply_fdia(sensor_data, target_bus=7, offset=0.15)

    if t == 5:
        break_communication(sensor_data, bus_id=15)

    simulate_info_layer(net, sensor_data)

    # Log metrics
    voltage_dev = compute_voltage_deviation(net, baseline_vm_pu)
    print(f"t={t}, Voltage Deviation: {voltage_dev:.4f}")

### Resilience Metrics Evaluation

In [26]:
# Ensure in_service flags exist and are True
for table in ["bus", "line", "gen", "load", "ext_grid", "trafo"]:
    if table in net:
        if "in_service" not in net[table].columns:
            net[table]["in_service"] = True
        else:
            net[table]["in_service"] = net[table]["in_service"].fillna(True)

# Ensure generator min/max P exist and are reasonable
if "gen" in net:
    if "min_p_mw" not in net.gen.columns:
        # Allow them to go down to 0 (or negative if you want)
        net.gen["min_p_mw"] = 0.0
    if "max_p_mw" not in net.gen.columns:
        # A bit of headroom above current dispatch
        net.gen["max_p_mw"] = net.gen["p_mw"].values * 1.2

# Test base-case power flow
pp.runpp(net)
print("Base case converged:", net.converged)

# =====================================================================================
# 2) GLOBAL CONFIG & RESILIENCE PARAMETERS
# =====================================================================================

case_name   = "RTS24_resilience"
n_ts        = 96             # 24h @ 15-min
step_hours  = 0.25           # 15 minutes
rng         = np.random.default_rng(0)

# Number of scenarios & PMUs for RTS24 (adjust as needed)
NUM_SCENARIOS = 10
NUM_PMUS      = 6

# Weights for cyber score
u1 = 0.3   # Weight for C_t
u2 = 0.25  # Weight for T_det
u3 = 0.25  # Weight for T_resp
u4 = 0.2   # Weight for T_rec

# Physical & cyber constraints
gamma = 0.85       # Minimum acceptable load service ratio during response
epsilon = 0.10     # Max voltage deviation during response (conceptual)
Beta = 0.85        # Minimum acceptable cyber reachability during recovery
Gamma = 0.95       # Load service recovery threshold
epsilon_rec = 0.05 # Voltage deviation during recovery (stricter)

# Time normalizations (hours)
Tdet_max = 6.0
Tresp_max = 4.0
Trec_max = 12.0

# Create base directory for this case
base_root = f"./{case_name}"
os.makedirs(base_root, exist_ok=True)

# Store original net state
original_line_in_service = net.line["in_service"].copy()
original_gen_in_service = net.gen["in_service"].copy() if "gen" in net else None
original_load_in_service = net.load["in_service"].copy() if "load" in net else None

# Scenario set
target_element  = "line"
element_indices = list(net[target_element].index)

# Time index
time_index = pd.date_range("2025-01-01 00:00:00", periods=n_ts,
                           freq=pd.Timedelta(hours=step_hours))

# =====================================================================================
# 3) HELPER FUNCTIONS
# =====================================================================================

def add_bus_information(net):
    """Attach bus-related info to result DataFrames (res_line, res_gen, etc.)."""
    if "res_line" in net and "line" in net:
        net.res_line = net.res_line.merge(
            net.line[["from_bus", "to_bus", "in_service"]],
            left_index=True, right_index=True, how="left", suffixes=('', '_y')
        )
        net.res_line = net.res_line.loc[:, ~net.res_line.columns.str.endswith('_y')]

    if "res_gen" in net and "gen" in net:
        net.res_gen = net.res_gen.merge(
            net.gen[["bus", "in_service"]],
            left_index=True, right_index=True, how="left", suffixes=('', '_y')
        )
        net.res_gen = net.res_gen.loc[:, ~net.res_gen.columns.str.endswith('_y')]

    if "res_load" in net and "load" in net:
        net.res_load = net.res_load.merge(
            net.load[["bus", "in_service"]],
            left_index=True, right_index=True, how="left", suffixes=('', '_y')
        )
        net.res_load = net.res_load.loc[:, ~net.res_load.columns.str.endswith('_y')]

    if "res_ext_grid" in net and "ext_grid" in net:
        net.res_ext_grid = net.res_ext_grid.merge(
            net.ext_grid[["bus", "in_service"]],
            left_index=True, right_index=True, how="left", suffixes=('', '_y')
        )
        net.res_ext_grid = net.res_ext_grid.loc[:, ~net.res_ext_grid.columns.str.endswith('_y')]

    if "res_trafo" in net and "trafo" in net:
        net.res_trafo = net.res_trafo.merge(
            net.trafo[["hv_bus", "lv_bus", "in_service"]],
            left_index=True, right_index=True, how="left", suffixes=('', '_y')
        )
        net.res_trafo = net.res_trafo.loc[:, ~net.res_trafo.columns.str.endswith('_y')]

    return net


def compute_load_served_ratio_from_results(net):
    """Compute LSR = Σ served_load / Σ total_demand based on active lines."""
    if not ("res_load" in net and "load" in net and "line" in net):
        return np.nan

    active_lines = net.line[net.line["in_service"] == True]
    active_buses = set(active_lines["from_bus"]).union(set(active_lines["to_bus"]))

    if "bus" not in net.res_load.columns:
        net = add_bus_information(net)

    served_loads = net.res_load[net.res_load["bus"].isin(active_buses)]
    served = served_loads["p_mw"].sum()

    demand = net.res_load["p_mw"].sum()
    if demand <= 0:
        return np.nan

    lsr = min(served / demand, 1.0)
    return lsr


def compute_voltage_deviation(net):
    """Compute maximum voltage deviation from 1.0 p.u."""
    if "res_bus" not in net:
        return np.nan
    voltage_deviations = np.abs(net.res_bus["vm_pu"] - 1.0)
    return voltage_deviations.max()


def compute_voltage_persistence_from_results(net):
    """Voltage persistence based on largest abnormal voltage cluster."""
    if "res_bus" not in net:
        return np.nan

    res_bus = net.res_bus
    abnormal = res_bus.index[np.abs(res_bus.vm_pu - 1.0) > 0.05].tolist()
    if not abnormal:
        return 1.0

    G = nx.Graph()
    for _, row in net.line.iterrows():
        if row["in_service"]:
            G.add_edge(int(row["from_bus"]), int(row["to_bus"]))

    sub_nodes = [n for n in abnormal if n in G]
    if not sub_nodes:
        return 1.0

    components = list(nx.connected_components(G.subgraph(sub_nodes)))
    largest_cluster = max(len(c) for c in components) if components else 0
    P_V = largest_cluster / len(net.bus)
    return np.exp(-P_V)


def generalized_power_mean(values, weights, p=1):
    """Weighted power mean (p=1 = weighted arithmetic mean)."""
    values = np.array(values, dtype=float)
    weights = np.array(weights, dtype=float)
    if np.any(np.isnan(values)):
        return np.nan
    if p == 0:
        return np.prod(values ** weights)
    return (np.sum(weights * (values ** p))) ** (1 / p)


def create_cyber_network_from_pmus(net, num_pmUs):
    """Create a cyber control network based on PMU installations."""
    in_service_buses = net.bus[net.bus["in_service"]].index.tolist()
    if len(in_service_buses) == 0:
        print("⚠️ Warning: No in-service buses found!")
        return nx.Graph(), []

    # Degree-based PMU placement
    bus_degrees = {}
    for bus in in_service_buses:
        degree = len(net.line[(net.line["from_bus"] == bus) |
                              (net.line["to_bus"] == bus)])
        bus_degrees[bus] = degree

    sorted_buses = sorted(bus_degrees.items(),
                          key=lambda x: x[1],
                          reverse=True)
    pmu_buses = [bus for bus, _ in sorted_buses[:min(num_pmUs, len(in_service_buses))]]

    control_graph = nx.Graph()
    control_graph.add_nodes_from(pmu_buses)

    for _, row in net.line.iterrows():
        if row["in_service"]:
            from_bus, to_bus = row["from_bus"], row["to_bus"]
            if from_bus in pmu_buses and to_bus in pmu_buses:
                control_graph.add_edge(from_bus, to_bus)

    return control_graph, pmu_buses


def compute_cyber_reachability(control_graph, failed_nodes):
    """C_t = |reachable nodes from control center| / |total control nodes|."""
    total_nodes = len(control_graph.nodes)
    if total_nodes == 0:
        return 1.0

    working_graph = control_graph.copy()
    working_graph.remove_nodes_from(failed_nodes)

    if len(working_graph.nodes) == 0:
        return 0.0

    control_center = min(control_graph.nodes())

    if control_center in failed_nodes or control_center not in working_graph:
        return 0.0

    try:
        reachable_nodes = nx.node_connected_component(working_graph, control_center)
        C_t = len(reachable_nodes) / total_nodes
    except Exception:
        C_t = 0.0

    return C_t


# =====================================================================================
# 4) DETECTION, RESPONSE, RECOVERY OPTIMIZATIONS
# =====================================================================================

def solve_detection_optimization(net, control_graph, failed_nodes, pmu_buses):
    """
    Detection time based on observability and cyber reachability.

    Returns T_det in hours (possibly np.inf if undetectable).
    """
    C_t = compute_cyber_reachability(control_graph, failed_nodes)

    if C_t < 0.3:
        return np.inf

    total_buses = len(net.bus)
    pmu_coverage = len(pmu_buses) / total_buses if total_buses > 0 else 0

    T_min_detect = 0.25  # 15 minutes minimum
    observability_score = C_t * pmu_coverage

    if observability_score < 0.1:
        return np.inf

    delay_factor = (1.0 - observability_score) * 2.0
    T_det = T_min_detect + delay_factor

    return T_det


def solve_response_optimization(net, t_attack, t_detect, gamma_param=0.85,
                                 epsilon_param=0.10, cyber_reachability=1.0):
    """
    Response time optimization using CVXPY.

    Returns T_resp in hours.
    """
    try:
        n_gens = len(net.gen) if "gen" in net else 0
        n_loads = len(net.load) if "load" in net else 0

        if n_gens == 0 or n_loads == 0:
            return 1.0

        P_gen = cp.Variable(n_gens)
        P_load = cp.Variable(n_loads)

        P_gen_max = net.gen["max_p_mw"].values
        P_gen_min = net.gen["min_p_mw"].values
        P_load_nom = net.load["p_mw"].values

        # ramp_capability = 0.5 + 0.5 * cyber_reachability  # reserved if needed

        objective = cp.Maximize(cp.sum(P_load))

        constraints = []
        constraints.append(cp.sum(P_load) >= gamma_param * np.sum(P_load_nom))
        constraints.append(P_gen >= P_gen_min)
        constraints.append(P_gen <= P_gen_max)
        constraints.append(P_load >= 0)
        constraints.append(P_load <= P_load_nom)
        constraints.append(cp.sum(P_gen) == cp.sum(P_load))

        problem = cp.Problem(objective, constraints)
        problem.solve(solver=cp.ECOS, verbose=False)

        if problem.status in [cp.OPTIMAL, cp.OPTIMAL_INACCURATE]:
            T_resp_base = 0.5
            cyber_delay = (1.0 - cyber_reachability) * 1.0

            load_served = cp.sum(P_load).value if cp.sum(P_load).value is not None else 0
            load_shed_fraction = 1.0 - (load_served / np.sum(P_load_nom))
            shedding_delay = load_shed_fraction * 0.5

            T_resp = T_resp_base + cyber_delay + shedding_delay
            return T_resp
        else:
            return 4.0

    except Exception as e:
        print(f"⚠️ Response optimization failed: {e}")
        return 2.0


def solve_recovery_optimization(net, control_graph, failed_nodes, T_resp,
                                Gamma_param=0.95, Beta_param=0.85,
                                epsilon_param=0.05, failed_lines=None):
    """
    Recovery time optimization using CVXPY.

    Returns T_rec in hours.
    """
    try:
        # --- Cyber repair time ---
        n_failed_cyber = len(failed_nodes)
        if n_failed_cyber == 0:
            T_cyber_recovery = 0.25
        else:
            repair_time_per_node = 0.5
            max_parallel_cyber_repairs = 3

            total_nodes = len(control_graph.nodes)
            nodes_needed = int(np.ceil(Beta_param * total_nodes))
            nodes_to_repair = min(nodes_needed, n_failed_cyber)

            T_cyber_recovery = (nodes_to_repair * repair_time_per_node) / max_parallel_cyber_repairs

        # --- Physical repair time ---
        n_failed_lines = len(failed_lines) if failed_lines else 1
        repair_time_per_line = 2.0
        max_parallel_line_repairs = 2

        repair_already_done = 0.5 * T_resp
        total_physical_repair_time = (n_failed_lines * repair_time_per_line) / max_parallel_line_repairs
        T_physical_recovery = max(0, total_physical_repair_time - repair_already_done)

        # --- Load restoration optimization ---
        n_loads = len(net.load) if "load" in net else 0
        n_gens = len(net.gen) if "gen" in net else 0

        if n_loads > 0 and n_gens > 0:
            try:
                P_load_restored = cp.Variable(n_loads)
                P_gen_dispatch = cp.Variable(n_gens)

                P_load_nom = net.load["p_mw"].values
                P_gen_max = net.gen["max_p_mw"].values
                P_gen_min = net.gen["min_p_mw"].values

                objective = cp.Maximize(cp.sum(P_load_restored))

                constraints = []
                constraints.append(cp.sum(P_load_restored) >= Gamma_param * np.sum(P_load_nom))
                constraints.append(P_gen_dispatch >= P_gen_min)
                constraints.append(P_gen_dispatch <= P_gen_max)
                constraints.append(P_load_restored >= 0)
                constraints.append(P_load_restored <= P_load_nom)
                constraints.append(cp.sum(P_gen_dispatch) == cp.sum(P_load_restored))

                problem = cp.Problem(objective, constraints)
                problem.solve(solver=cp.ECOS, verbose=False)

                if problem.status in [cp.OPTIMAL, cp.OPTIMAL_INACCURATE]:
                    T_load_restoration = (Gamma_param - gamma) / 0.1 * 0.5
                else:
                    T_load_restoration = 1.0
            except Exception:
                T_load_restoration = 0.5
        else:
            T_load_restoration = 0.5

        T_hold = 1.0

        parallel_phase = max(T_cyber_recovery, T_physical_recovery * 0.7)
        sequential_phase = T_physical_recovery * 0.3 + T_load_restoration + T_hold

        T_rec = parallel_phase + sequential_phase
        T_rec = max(T_rec, 1.5)

        return T_rec

    except Exception as e:
        print(f"⚠️ Recovery optimization failed: {e}")
        return max(len(failed_nodes) * 0.5 + 2.0, 2.0)


def compute_cyber_score(C_t, T_det, T_resp, T_rec):
    """
    Cyber score:
    S_C = C_t^u1 * exp(-T_det/Tdet_max)^u2 *
          exp(-T_resp/Tresp_max)^u3 * exp(-T_rec/Trec_max)^u4
    """
    if T_det is None or np.isinf(T_det):
        return 0.0

    T_det_use = T_det if T_det is not None else 0.0
    T_resp_use = T_resp if T_resp is not None else 0.0
    T_rec_use = T_rec if T_rec is not None else 0.0

    T_det_use = min(T_det_use, Tdet_max)
    T_resp_use = min(T_resp_use, Tresp_max)
    T_rec_use = min(T_rec_use, Trec_max)

    term1 = C_t ** u1
    term2 = np.exp(-T_det_use / Tdet_max) ** u2
    term3 = np.exp(-T_resp_use / Tresp_max) ** u3
    term4 = np.exp(-T_rec_use / Trec_max) ** u4

    SC = term1 * term2 * term3 * term4
    return SC


def compute_R_t(SP, SC, eta=0.2):
    """
    System-level resilience:
    R(t) = SP(t) * [η + (1-η) * √SC(t)]
    """
    if np.isnan(SP) or np.isnan(SC):
        return np.nan
    R = SP * (eta + (1 - eta) * np.sqrt(SC))
    return R


# =====================================================================================
# 5) PRECOMPUTE PMU-BASED CYBER NETWORK FOR RTS24
# =====================================================================================

control_graph, pmu_buses = create_cyber_network_from_pmus(net, NUM_PMUS)
print("PMU buses (RTS24):", pmu_buses)

# Baseline detection capability (no failures)
T_det_baseline = solve_detection_optimization(net, control_graph, [], pmu_buses)
print(f"Baseline T_det (no attack): {T_det_baseline:.2f} h" if not np.isinf(T_det_baseline)
      else "Baseline T_det: undetectable")

# =====================================================================================
# 6) MAIN SIMULATION LOOP OVER SCENARIOS
# =====================================================================================

for s_idx in range(1, NUM_SCENARIOS + 1):
    # Restore original in_service flags
    net.line["in_service"] = original_line_in_service.copy()
    if original_gen_in_service is not None:
        net.gen["in_service"] = original_gen_in_service.copy()
    if original_load_in_service is not None:
        net.load["in_service"] = original_load_in_service.copy()

    # Scenario directories
    scenario_root = os.path.join(base_root, f"scenario{s_idx}")
    os.makedirs(scenario_root, exist_ok=True)
    os.makedirs(os.path.join(scenario_root, "logs"), exist_ok=True)
    os.makedirs(os.path.join(scenario_root, "power_flow_results"), exist_ok=True)

    # Random attack time and line
    trigger_step = random.randint(int(0.2 * n_ts), int(0.4 * n_ts))
    target_line_idx = random.choice(element_indices)

    meta_rows = [{
        "scenario": s_idx,
        "target_element": target_element,
        "target_index": int(target_line_idx),
        "planned_trigger_step": int(trigger_step),
        "attack_fired": False,
        "T_det_hours": None,
        "T_resp_hours": None,
        "T_rec_hours": None,
    }]

    failed_nodes = []
    failed_lines = []
    resilience_records = []
    attack_fired = False

    # Initialize T_det, T_resp, T_rec before attack
    T_det = T_det_baseline
    T_resp = 0.0
    T_rec = 0.0

    for t, ts in enumerate(tqdm(time_index, desc=f"Scenario {s_idx}", leave=False)):
        # Trigger attack
        if t == trigger_step and not attack_fired:
            if net.line.at[target_line_idx, "in_service"]:
                # ---- Physical line outage ----
                net.line.at[target_line_idx, "in_service"] = False
                failed_lines = [target_line_idx]

                # ---- Cyber node failures (20% of PMUs) ----
                num_failed = max(1, int(0.2 * len(control_graph.nodes())))
                failed_nodes = random.sample(list(control_graph.nodes()), k=num_failed)

                attack_fired = True
                meta_rows[0]["attack_fired"] = True

                print(f"\n  ⚡ Attack in Scenario {s_idx} at t={t}: "
                      f"Line {target_line_idx}, {num_failed} cyber nodes")

                # ---- Recompute T_det with failed nodes ----
                print("  🔍 Computing T_det...")
                T_det = solve_detection_optimization(net, control_graph, failed_nodes, pmu_buses)
                meta_rows[0]["T_det_hours"] = float(T_det) if not np.isinf(T_det) else None

                if not np.isinf(T_det):
                    print(f"     T_det = {T_det:.2f} hours")

                    # ---- Compute T_resp ----
                    print("  ⚙️ Computing T_resp...")
                    C_t_now = compute_cyber_reachability(control_graph, failed_nodes)
                    T_resp = solve_response_optimization(
                        net, trigger_step,
                        trigger_step + int(T_det / step_hours),
                        gamma_param=gamma, epsilon_param=epsilon,
                        cyber_reachability=C_t_now
                    )
                    meta_rows[0]["T_resp_hours"] = float(T_resp)
                    print(f"     T_resp = {T_resp:.2f} hours")

                    # ---- Compute T_rec ----
                    print("  🔄 Computing T_rec...")
                    T_rec = solve_recovery_optimization(
                        net, control_graph, failed_nodes, T_resp,
                        Gamma_param=Gamma, Beta_param=Beta,
                        epsilon_param=epsilon_rec, failed_lines=failed_lines
                    )
                    meta_rows[0]["T_rec_hours"] = float(T_rec)
                    print(f"     T_rec = {T_rec:.2f} hours")
                    print(f"     TOTAL = {T_det + T_resp + T_rec:.2f} hours\n")
                else:
                    print("     Attack undetectable!\n")
                    T_resp = Tresp_max
                    T_rec = Trec_max

        # Run power flow
        try:
            pp.runpp(net, algorithm="nr", calculate_voltage_angles=True,
                     enforce_q_lims=True)
            converged = net.converged
        except Exception:
            converged = False

        if not converged:
            SP_t = SC_t = R_t = LSR_t = f2_t = C_t_val = np.nan
        else:
            net = add_bus_information(net)

            # Physical layer
            LSR_t = compute_load_served_ratio_from_results(net)
            f2_t = compute_voltage_persistence_from_results(net)
            SP_t = generalized_power_mean([LSR_t, f2_t],
                                          weights=[0.6, 0.4], p=1)

            # Cyber layer
            C_t_val = compute_cyber_reachability(control_graph, failed_nodes)

            # Cyber score and resilience
            SC_t = compute_cyber_score(C_t_val, T_det, T_resp, T_rec)
            R_t = compute_R_t(SP_t, SC_t)

        resilience_records.append({
            "time": ts,
            "timestep": t,
            "LSR": LSR_t,
            "Voltage_Persistence": f2_t,
            "SP": SP_t,
            "C_t": C_t_val,
            "T_det": T_det,
            "T_resp": T_resp,
            "T_rec": T_rec,
            "SC": SC_t,
            "R": R_t,
            "converged": converged,
            "attack_active": attack_fired,
        })

    # Store results for this scenario
    resilience_df = pd.DataFrame(resilience_records)
    resilience_df.to_csv(os.path.join(scenario_root, "resilience_timeseries.csv"),
                         index=False)
    pd.DataFrame(meta_rows).to_csv(os.path.join(scenario_root, "logs",
                                                "scenario_meta.csv"),
                                   index=False)

    pf_results_dir = os.path.join(scenario_root, "power_flow_results")
    if "res_load" in net:
        net.res_load.to_csv(os.path.join(pf_results_dir, "res_load.csv"))
    if "res_line" in net:
        net.res_line.to_csv(os.path.join(pf_results_dir, "res_line.csv"))
    if "res_bus" in net:
        net.res_bus.to_csv(os.path.join(pf_results_dir, "res_bus.csv"))
    if "res_gen" in net:
        net.res_gen.to_csv(os.path.join(pf_results_dir, "res_gen.csv"))
    if "res_ext_grid" in net:
        net.res_ext_grid.to_csv(os.path.join(pf_results_dir, "res_ext_grid.csv"))
    if "res_trafo" in net:
        net.res_trafo.to_csv(os.path.join(pf_results_dir, "res_trafo.csv"))

    # Save PMU configuration (fixed for all scenarios, but kept per scenario)
    pmu_config = pd.DataFrame({
        "pmu_bus": pmu_buses,
        "is_control_center": [bus == min(pmu_buses) for bus in pmu_buses]
                              if len(pmu_buses) > 0 else []
    })
    pmu_config.to_csv(os.path.join(scenario_root, "logs", "pmu_configuration.csv"),
                      index=False)

    print(f"✅ Scenario {s_idx} completed\n")

# Restore original state
net.line["in_service"] = original_line_in_service.copy()
if original_gen_in_service is not None:
    net.gen["in_service"] = original_gen_in_service.copy()
if original_load_in_service is not None:
    net.load["in_service"] = original_load_in_service.copy()

# Zip all results
zip_path = shutil.make_archive(case_name, "zip", root_dir=".", base_dir=case_name)
print(f"✅ Done. All scenarios saved in '{case_name}.zip'")

')#' is not recognized as an internal or external command,
operable program or batch file.


Base case converged: True
PMU buses (RTS24): [20, 14, 15, 19, 22, 0]
Baseline T_det: undetectable


Scenario 1:  29%|██▉       | 28/96 [00:00<00:02, 29.64it/s]


  ⚡ Attack in Scenario 1 at t=23: Line 32, 1 cyber nodes
  🔍 Computing T_det...
     Attack undetectable!



✅ Scenario 1 completed



...


  ⚡ Attack in Scenario 9 at t=29: Line 6, 1 cyber nodes
  🔍 Computing T_det...
     Attack undetectable!



✅ Scenario 9 completed



Scenario 10:  33%|███▎      | 32/96 [00:00<00:02, 31.85it/s]


  ⚡ Attack in Scenario 10 at t=26: Line 29, 1 cyber nodes
  🔍 Computing T_det...
     Attack undetectable!



✅ Scenario 10 completed

✅ Done. All scenarios saved in 'RTS24_resilience.zip'
